CLUSTERING CLIENTS

In [1]:
import pandas as pd 
import numpy as np 
import time 

from sklearn.base import BaseEstimator, TransformerMixin 
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline 
from sklearn.cluster import KMeans 
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split 
from sklearn.metrics import silhouette_score

from sklearn import metrics
from sklearn.utils import shuffle
from sklearn.preprocessing import normalize

import os

In [2]:
ruta= os.path.join(os.getcwd())
ruta

'D:\\Muñoz\\Cursos\\Python\\Todoterreno\\CursoPython\\Master\\Nuevo\\Proyectos\\Segmentación'

In [3]:
df_clientes=pd.read_csv(ruta+'\..\data\clientes.csv', sep=',')
df_pagos=pd.read_csv(ruta+'\..\data\pagos.csv', sep=',')
df_pedidos=pd.read_csv(ruta+'\..\data\pedidos.csv', sep=',')

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
C:\Users\Carlos\AppData\Local\Temp\ipykernel_12560\722115688.py:1: SyntaxWarning: invalid escape sequence '\.'
  df_clientes=pd.read_csv(ruta+'\..\data\clientes.csv', sep=',')
C:\Users\Carlos\AppData\Local\Temp\ipykernel_12560\722115688.py:2: SyntaxWarning: invalid escape sequence '\.'
  df_pagos=pd.read_csv(ruta+'\..\data\pagos.csv', sep=',')
C:\Users\Carlos\AppData\Local\Temp\ipykernel_12560\722115688.py:3: SyntaxWarning: invalid escape sequence '\.'
  df_pedidos=pd.read_csv(ruta+'\..\data\pedidos.csv', sep=',')
C:\Users\Carlos\AppData\Local\Temp\ipykernel_12560\722115688.py:1: SyntaxWarning: invalid escape sequence '\.'
  df_clientes=pd.read_csv(ruta+'\..\data\clientes.csv', sep=',')
C:\U

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Muñoz\\Cursos\\Python\\Todoterreno\\CursoPython\\Master\\Nuevo\\Proyectos\\Segmentación\\..\\data\\clientes.csv'

In [ ]:
df_clientes.head()

In [ ]:
df_pagos.head()

In [ ]:
df_pedidos.head()

### EDA, CLEANING AND PREPROCESSING

In [ ]:
import sys
# Add the path to the main directory to the sys.path
sys.path.append(os.path.abspath('..'))

In [ ]:
from utils import data_report

data_report(df_clientes)

In [ ]:
data_report(df_pagos)

In [ ]:
data_report(df_pedidos)

### GENERATE NEW AGGREGATE VARIABLES

In [ ]:
df_pagos

In [ ]:
df_pagos['payment_type'].value_counts()

In [ ]:
# 2. Creating a Boolean column for credit card payments
# This column will indicate True if the payment was made by credit card, False otherwise
df_pagos["boolean_credit_card"] = df_pagos["payment_type"] == 'credit_card'

In [ ]:
# 3. Ordinal coding function for payment types
# A numeric value is assigned based on the combination of payment methods
def ordinal_encoding(columna):
    string=str(set(columna))
    if "credit_card" in string and "voucher" in string:
        return 3
    elif "credit_card" in string and "debit_card" in string:
        return 2
    else:
        return 1

In [ ]:
# 4. Aggregation of payments per order
# We calculate sums, counts, and maximum payments for each order
gbdf_ = df_pagos.groupby(["order_id"]).agg(
    total_sum=("payment_value", np.sum),  # Total amount of payments
    total_credit_card_operations=("boolean_credit_card", np.sum),  # Total payments made with credit card
    max_payments_sequential=("payment_sequential", np.max),  # Maximum sequential number of payments
    unique_payments_types=("payment_type", lambda series: str(set(series))),  # Payment methods
    ordinal_var=("payment_type", ordinal_encoding)  # Ordinal coding of payments
)


In [ ]:
# 5. Aggregation of general payments by order

aggregate_payments = df_pagos.groupby(['order_id']).agg(
    max_pay=("payment_value", "max"),  # Maximum payment made in one order
    min_pay=("payment_value", "min"),  # Minimum payment made on an order
    mean_pay=("payment_value", "mean"), # Average payment per order
    total_pay=("payment_value", "sum"), # Total payment per order
    max_seq=("payment_sequential", "max"),  # Maximum sequential payment value
)

In [ ]:
df_pedidos.head()

In [ ]:
# 6. Merge orders with payments
# We merge the payment information with the order table

orders_with_payments = pd.merge(df_pedidos, aggregate_payments, on='order_id')
orders_with_payments

In [ ]:
# 7. Conversión de fechas a formato datetime
# Se convierten las columnas de fecha para facilitar los cálculos temporales
orders_with_payments['order_purchase_timestamp'] = pd.to_datetime(orders_with_payments['order_purchase_timestamp'], format='%Y-%m-%d %H:%M:%S')

orders_with_payments['order_delivered_customer_date'] = pd.to_datetime(orders_with_payments['order_delivered_customer_date'], format='%Y-%m-%d %H:%M:%S')

orders_with_payments["order_estimated_delivery_date"] = pd.to_datetime(orders_with_payments['order_estimated_delivery_date'], format='%Y-%m-%d %H:%M:%S')

In [ ]:
orders_with_payments.info()

In [ ]:
# 8. Calculation of new temporary variables
orders_with_payments["last_purchase"] = orders_with_payments["order_purchase_timestamp"].max()
orders_with_payments["time_since_last_purchase"] = orders_with_payments["last_purchase"] - orders_with_payments["order_purchase_timestamp"]
orders_with_payments["delivery_time"] = orders_with_payments["order_delivered_customer_date"] - orders_with_payments["order_purchase_timestamp"]
orders_with_payments["delay"] = orders_with_payments["order_delivered_customer_date"] - orders_with_payments["order_estimated_delivery_date"]

In [ ]:
# 9. Merging with customer information
df_final=pd.merge(df_clientes, orders_with_payments, on='customer_id')

In [ ]:
df_final

In [ ]:
df_pagos.head()

In [ ]:
df_pedidos.head()

In [ ]:
df_clientes.head()

In [ ]:
df_final.set_index('customer_unique_id', inplace=True)
df_final.head()

In [ ]:
lista_relevantes = [
    'max_pay', 'min_pay', 'mean_pay', 'total_pay', 'max_seq',
    'time_since_last_purchase', 'delivery_time', 'delay'
]

df_final = df_final[lista_relevantes]
df_final.head()

In [ ]:
df_final.describe()

PIPELINE

In [ ]:
class ArrayToDataFrame(BaseEstimator,TransformerMixin):
    """Transforma un array en un DataFrame."""
    def __init__(self, columns, index=None):
        self.columns=columns
        self.index=index

    def fit(self,X,y=None):
        return self

    def transform(self,X,y=None):
        if self.index is not None:
            df = pd.DataFrame(X, columns=self.columns, index=self.index)
        else:
            df = pd.DataFrame(X, columns=self.columns)
        return df

In [ ]:
class FeatureGenerator(BaseEstimator,TransformerMixin):
    '''
    Genera nuevas características a partir de datos existentes.
    '''
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        aggregated_df = X.groupby(X.index).agg(
            n_orders=('total_pay', 'count'),
            amount=('total_pay', 'sum'),
            avg_ticket=('total_pay', 'mean'),
            last_purchase=('time_since_last_purchase', 'min'),
            first_purchase=('time_since_last_purchase', 'max'),
            mean_delivery_time=('delivery_time', 'mean'),
            max_delivery_time=('delivery_time', 'max'),
            mean_delay=('delay', 'mean'),
            max_delay=('delay', 'max')
        )
        return aggregated_df

In [ ]:
class OutlierFilter(BaseEstimator,TransformerMixin):
    '''
    Filtra outliers utilizando cuantiles.
    '''
    def __init__(self,q,col_to_filter):
        self.q=q
        self.col_to_filter=col_to_filter
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        criteria_list = []
        for col in self.col_to_filter:
            criteria = X[col] < np.quantile(X[col], q=self.q)
            criteria_list.append(criteria)

        global_criteria = criteria_list[0]
        for criteria in criteria_list[1:]:
            global_criteria = global_criteria & criteria

        X = X[global_criteria]
        self.index = X.index
        return X
    

In [ ]:
df_final=df_final.sample(10000)

In [ ]:
df_final.head()

In [ ]:
columns=list(df_final.columns)
index=list(df_final.index)

In [ ]:
df_final["time_since_last_purchase"] = df_final["time_since_last_purchase"] / np.timedelta64(1, "D")
df_final["delivery_time"] = df_final["delivery_time"] / np.timedelta64(1, "D")
df_final["delay"] = df_final["delay"] / np.timedelta64(1, "D")


In [ ]:
pipe = Pipeline(
    steps= [
        # Fill in the nulls
        ('Imputer', KNNImputer()),
        # Transform from array to dataset
        ('ArrayToDataFrame', ArrayToDataFrame(columns, index=index)),
        # Create the variables
        ('FeatureGenerator', FeatureGenerator()),
        # Remove outliers, remember millionaire clients
        ("OutlierFilter", OutlierFilter(q=0.99, col_to_filter=["amount", "max_delay"])),
        # Scaling data to fit it into clustering algorithms
        ("StandartScaler", StandardScaler())
    ]
)

In [ ]:
df_scaled_transformed_no_outliers = pipe.fit_transform(df_final)
df_scaled_transformed_no_outliers

K-MEANS MODELS, Gaussian Mixtures (Probabilistic), Agglomerative Hierarchical Clustering (Hierarchical)

KMEANS

In [ ]:
# cluster search
elbow = True
if elbow:
    sse={}
    for k in range(2,15):
        print(f"Los datos se ajustan {k} clusters")
        kmeans=KMeans(n_clusters=k)
        kmeans.fit(df_scaled_transformed_no_outliers)
        sse[k]=kmeans.inertia_


In [ ]:
sse

In [ ]:
if elbow:
    fig = plt.figure(figsize = (16, 8))
    ax=fig.add_subplot()

    x=list(sse.keys())
    y=list(sse.values())
    ax.plot(x, y, label= "Inercia")
    fig.suptitle("k")


Kmeans

In [ ]:
pipe = Pipeline(
    steps= [
        # Fill in the nulls
        ('Imputer', KNNImputer()),
        # Transform from array to dataset
        ('ArrayToDataFrame', ArrayToDataFrame(columns, index=index)),
        # Create the variables
        ('FeatureGenerator', FeatureGenerator()),
        # Remove outliers, remember millionaire clients
        ("OutlierFilter", OutlierFilter(q=0.99, col_to_filter=["amount", "max_delay"])),
        # Scaling data to fit it into clustering algorithms
        ("StandartScaler", StandardScaler()),
        #Create clusters, small groups of customers
        ("Clustering", KMeans(n_clusters=5))

    ]
)

In [ ]:
pipe.fit(df_final)

In [ ]:
centroides=pipe['Clustering'].cluster_centers_

In [ ]:
X_processed_km = pipe[:3].transform(df_final)
X_scaled = pipe["StandartScaler"].transform(X_processed_km)
labels = pipe["Clustering"].predict(X_scaled)
X_processed_km["cluster"] = labels

In [ ]:
X_processed_km

### PROBABILISTIC
Cluster search, Silhouette method

In [ ]:
for i in df_final.columns:
    sns.kdeplot(df_final[i], fill=True)
    plt.show()

In [ ]:
cov_matrix=df_final.cov()
plt.figure(figsize=(20,8))
sns.heatmap(cov_matrix,annot=True,cmap='coolwarm',fmt='.2f')
plt.title('Matriz de covarianza')
plt.show()

#  If the variables are highly correlated, the resulting Gaussian distributions will be more elliptical in the feature space, which influences the estimation of the model parameters.

The Gaussian mixtures algorithm is not the most suitable for this type of data, as the variables do not follow a normal distribution. However, we will normalize the data in the pipeline to be able to perform the modeling.
The covariance-type parameter, in this case, will be set to full. Since the variables are disparate, there is high covariance in max, min, mean, and total, and negative covariance in delay and time-since-last-purchase.

In [ ]:
def SelBest(arr:list, X:int)->list:

  dx=np.argsort(arr)[:X]
  return arr[dx]

n_clusters=np.arange(2,13)
sils=[]
sils_err=[]
iterations=7
for n in n_clusters:
  tmp_sil=[]
  for n in range(2, 11):
      gmm = GaussianMixture(n_components=n, n_init=2, covariance_type='full').fit(df_scaled_transformed_no_outliers)
      labels = gmm.predict(df_scaled_transformed_no_outliers)
      sil = metrics.silhouette_score(df_scaled_transformed_no_outliers, labels, metric='euclidean')
      tmp_sil.append(sil)

  val = np.mean(SelBest(np.array(tmp_sil), int(iterations/5)))
  err=np.std(tmp_sil)
  sils.append(val)
  sils_err.append(err)

plt.errorbar(n_clusters,sils,yerr=sils_err)
plt.title('silhouette_score',fontsize=15)
plt.xticks(n_clusters)
plt.xlabel('n_clusters')
plt.ylabel('score')

The purpose of this function, in the context of the code, is to select the best results (in this case, the best silhouette scores) from a set of simulations or repetitions to obtain a more reliable estimate of the model's quality.

In [ ]:

def gmm_js(gmm_p, gmm_q, n_samples=10000):
    X = gmm_p.sample(n_samples)[0]
    log_p_X = gmm_p.score_samples(X)
    log_q_X = gmm_q.score_samples(X)
    log_mix_X = np.logaddexp(log_p_X, log_q_X)

    Y = gmm_q.sample(n_samples)[0]
    log_p_Y = gmm_p.score_samples(Y)
    log_q_Y = gmm_q.score_samples(Y)
    log_mix_Y = np.logaddexp(log_p_Y, log_q_Y)

    js_divergence = (log_p_X.mean() - (log_mix_X.mean() - np.log(2)) + log_q_Y.mean() - (log_mix_Y.mean() - np.log(2))) / 2
    return np.sqrt(js_divergence)

def select_best(dist, k):
    return np.sort(dist)[:k]

n_clusters = np.arange(2, 13)
iterations = 7
results = []
res_sigs = []

for n in n_clusters:
    dist = []
    for iteration in range(iterations):
        train, test = train_test_split(df_scaled_transformed_no_outliers, test_size=0.5)

        gmm_train = GaussianMixture(n_components=n, n_init=2, covariance_type='full').fit(train)
        gmm_test = GaussianMixture(n_components=n, n_init=2, covariance_type='full').fit(test)
        dist.append(gmm_js(gmm_train, gmm_test))

    selec = select_best(np.array(dist), int(iterations / 5))
    result = np.mean(selec)
    res_sig = np.std(selec)
    results.append(result)
    res_sigs.append(res_sig)

plt.errorbar(n_clusters, results, yerr=res_sigs)
plt.xlabel('Number of clusters')
plt.ylabel('Jensen-Shannon Divergence')
plt.title('GMM JS Divergence vs Number of Clusters')
plt.show()

The JS divergence distances between the trained GMM models and the test data are calculated, along with their average and standard deviation. This value is used to assess the model's stability and generalizability. A lower divergence value suggests that the model has better generalizability, meaning it is less likely to overfit the data.

Chart: A lower Jensen-Shannon divergence number might suggest that the model with that specific number of clusters is more consistent across different data splits.
A model is created for each cluster, and the Jensen-Shannon divergence is calculated.

The purpose of the functions and graphs in this code is to evaluate the quality and stability of GMM models under different configurations. The Silhouette Score provides a measure of cluster cohesion, while the Jensen-Shannon Divergence assesses the model's generalizability. Together, these approaches help identify the most appropriate number of clusters and ensure that the model is not overfitted to the training data.

## Gaussian Model

In [ ]:
pipe_gau = Pipeline(
    steps= [
        # Fill in the nulls
        ('Imputer', KNNImputer()),
        # Transform from array to dataset
        ('ArrayToDataFrame', ArrayToDataFrame(columns, index=index)),
        # Create the variables
        ('FeatureGenerator', FeatureGenerator()),
        # Remove outliers, remember millionaire clients
        ("OutlierFilter", OutlierFilter(q=0.99, col_to_filter=["amount", "max_delay"])),
        # Scaling data to fit it into clustering algorithms
        ("StandartScaler", StandardScaler()),
        #Create clusters, customer groups
        ("Clustering_Gaussian", GaussianMixture(n_components=6,n_init=2,covariance_type='diag'))

    ])

In [ ]:
X_processed_df_gau = pipe_gau[:3].fit_transform(df_final)
X_scaled = pipe_gau["StandartScaler"].fit_transform(X_processed_df_gau)
labels = pipe_gau["Clustering_Gaussian"].fit_predict(X_scaled)
X_processed_df_gau["cluster"] = labels
X_processed_df_gau

### JERARQUICO
Búsqueda de clusters

In [ ]:
linked = linkage(df_scaled_transformed_no_outliers, method='ward')
n= df_scaled_transformed_no_outliers.shape[0]
labellist=range(1,n+1)

plt.figure(figsize=(20, 8))
dendrogram(
    linked, 
    orientation='top',
    labels=labellist,
    distance_sort='descending',
    show_leaf_counts=True,
)

# Add the horizontal line
optimal_distance = 70
plt.axhline(y=optimal_distance, color='r', linestyle='--')

plt.title('Dendrograma con Línea Horizontal')
plt.xlabel('Índice de Muestra')
plt.ylabel('Distancia')
plt.show()

### How to decide on clusters?

Observe the top clusters.

According to chatGPT (One of the most common methods is to visually inspect the dendrogram to find the greatest vertical distance without horizontal line crossings. This point generally corresponds to an optimal dendrogram cut.)
A horizontal line is drawn on the graph to delimit the clusters, and the clusters are counted. The line is random.

Modelo Hierarchy

In [ ]:
pipe_hie = Pipeline(
    steps= [
        # Fill in the nulls
        ('Imputer', KNNImputer()),
        # Transform from array to dataset
        ('ArrayToDataFrame', ArrayToDataFrame(columns, index=index)),
        # Create the variables
        ('FeatureGenerator', FeatureGenerator()),
        # Remove outliers, remember millionaire clients
        ("OutlierFilter", OutlierFilter(q=0.99, col_to_filter=["amount", "max_delay"])),
        # Scaling data to fit it into clustering algorithms
        ("StandartScaler", StandardScaler()),
        #Create clusters, customer groups
        ("Clustering_Hierarchy", AgglomerativeClustering(n_clusters=6, linkage='ward'))

    ])

In [ ]:
X_processed_df_hie = pipe_hie[:3].fit_transform(df_final)
X_scaled = pipe_hie["StandartScaler"].fit_transform(X_processed_df_hie)
labels = pipe_hie["Clustering_Hierarchy"].fit_predict(X_scaled)
X_processed_df_hie["cluster"] = labels
X_processed_df_hie

### CONCLUSIONS
kmeans

In [ ]:
X_processed_km

In [ ]:
ficha_df=pd.DataFrame()

In [ ]:
for i, col in enumerate(['amount', 'n_orders', 'last_purchase', 'mean_delay']):
    resumen_data=X_processed_km[['cluster', col]].groupby('cluster').describe().T[1:]
    ficha_df = pd.concat([ficha_df, resumen_data])

In [ ]:
ficha_df

In [ ]:
out_index= [
    'Monetarios',
    "Fidelización",
    "Fidelización",
    "Logística"
]
inner_index=[
    "Importe",
    "Nr. de compras",
    "Última compra",
    "Retrasos"
]
estadisticos = ["Media", "Desviación", "Mínimo", "Perc. 25", "Perc. 50", "Perc. 75", "Máximo"]

new_multi_index=[]

for oi, ii in zip(out_index, inner_index):
    for es in estadisticos:
        new_multi_index.append((oi,ii,es))

new_multi_index

In [ ]:
def generate_multiindex(list_of_tuples, names):
    return pd.MultiIndex.from_tuples(list_of_tuples, names = names)


In [ ]:
names = ["Grupo Indicadores", "Indicador", "Estadístico"]
index_ficha = generate_multiindex(new_multi_index, names)
ficha_df.set_index(index_ficha, inplace = True)

In [ ]:
ficha_df

In [ ]:
tamaño_clusters = X_processed_km.groupby('cluster').size().to_frame().T 
tamaño_clusters.set_index(generate_multiindex([("General", "Clúster", "Tamaño")] , names), inplace = True)

In [ ]:
ficha_df = pd.concat([tamaño_clusters, ficha_df])
ficha_df

In [ ]:
ficha_df.style.background_gradient(cmap = 'Blues', axis = 1)